# Computational cost and model configuration

Fit, calibration and inference times describe separate operations. Tree structure counts and neural parameter counts measure different aspects of model size.

In [1]:
from pathlib import Path
import sys
_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'dataset').is_dir() and (p / 'notebooks').is_dir())
if str(_root / 'notebooks') not in sys.path:
    sys.path.insert(0, str(_root / 'notebooks'))
from _runtime import (reuse as _nb_reuse, source_document as _source_document,
                      initialize, repo_root as _repo_root, result_tables,
                      result_state, plot_data, figure_stem)
initialize()

from pathlib import Path
import json, hashlib, time, copy, os, threading
import numpy as np
import pandas as pd
import joblib, psutil
from IPython.display import display, Markdown
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from quantile_forest import RandomForestQuantileRegressor
from threadpoolctl import threadpool_limits
PROJECT_DIR=_repo_root()
TABLE_DIR,STATE_DIR,MODEL_DIR,PREDICT_DIR=[PROJECT_DIR/x for x in ['.cache/experiment/tables','.cache/experiment/state','.cache/experiment/models','.cache/experiment/predictions']]
for directory in [TABLE_DIR,STATE_DIR,MODEL_DIR,PREDICT_DIR]:directory.mkdir(exist_ok=True)
protocol=json.loads((STATE_DIR/'experiment_config.json').read_text())
assert json.loads((STATE_DIR/'temporal_consistency.json').read_text())['status']=='PASS'
modeling=pd.read_parquet(TABLE_DIR/'modeling_dataset.parquet')
feature_cols=json.loads((STATE_DIR/'main_preprocessing_state.json').read_text())['feature_columns']
pd.set_option('display.max_rows',100);pd.set_option('display.max_columns',20);pd.set_option('display.max_colwidth',None)
threadpool_limits(protocol['threads'])

base_panel=pd.read_parquet(TABLE_DIR/'causal_base_panel.parquet')
origins=pd.read_csv(TABLE_DIR/'rolling_origin_splits.csv',parse_dates=['start','end'])

import matplotlib.pyplot as plt
FIGURE_DIR=PROJECT_DIR/'.cache/experiment/figures'
plt.rcParams.update({'font.family':'Times New Roman','font.size':18,'axes.titlesize':20,'axes.labelsize':18,'xtick.labelsize':18,'ytick.labelsize':18,'legend.fontsize':18,'figure.dpi':350,'savefig.dpi':350})


## Reproducibility and measured runtime
All times come from executed fits, feature/graph construction, dedicated calibration or inference. Fresh fits and identical-data deterministic cache hits are separate. Tree node counts describe ensemble size; they are not neural trainable parameter counts. RSS is sampled process memory, so it includes the Python kernel and may miss peaks shorter than the sampling interval.

In [1]:
from pathlib import Path
import json, hashlib, time, copy, os, threading
import numpy as np
import pandas as pd
import joblib, psutil
from IPython.display import display, Markdown
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from quantile_forest import RandomForestQuantileRegressor
from threadpoolctl import threadpool_limits
PROJECT_DIR=_repo_root()
TABLE_DIR,STATE_DIR,MODEL_DIR,PREDICT_DIR=[PROJECT_DIR/x for x in ['.cache/experiment/tables','.cache/experiment/state','.cache/experiment/models','.cache/experiment/predictions']]
for directory in [TABLE_DIR,STATE_DIR,MODEL_DIR,PREDICT_DIR]:directory.mkdir(exist_ok=True)
protocol=json.loads((STATE_DIR/'experiment_config.json').read_text())
assert json.loads((STATE_DIR/'temporal_consistency.json').read_text())['status']=='PASS'
modeling=pd.read_parquet(TABLE_DIR/'modeling_dataset.parquet')
feature_cols=json.loads((STATE_DIR/'main_preprocessing_state.json').read_text())['feature_columns']
pd.set_option('display.max_rows',100);pd.set_option('display.max_columns',20);pd.set_option('display.max_colwidth',None)
threadpool_limits(protocol['threads'])

base_panel=pd.read_parquet(TABLE_DIR/'causal_base_panel.parquet')
origins=pd.read_csv(TABLE_DIR/'rolling_origin_splits.csv',parse_dates=['start','end'])

import sys,platform,importlib.metadata,subprocess
from threadpoolctl import threadpool_info
assert (STATE_DIR/'statistical_testing_complete.json').exists()


In [2]:
_nb_reuse('src-03-003', globals(), names=['load_model'])


In [4]:
runtime_files={'baseline_fresh_fit_runtime.csv':'main_tree_selection','ia_gtpf_fresh_fit_runtime.csv':'main_ia_selection','rolling_origin_fresh_fit_runtime.csv':'rolling','ablation_component_runtime.csv':'ablation','supplementary_public_fit_runtime.csv':'public_supplement'}
parts=[]
for filename,scope in runtime_files.items():
    path=TABLE_DIR/filename
    if path.exists():
        frame=pd.read_csv(path);frame['scope']=scope;frame['source_table']=filename;parts.append(frame)
components=pd.concat(parts,ignore_index=True);components.to_csv(TABLE_DIR/'runtime_component_ledger.csv',index=False)
main=components.loc[components.scope.isin(['main_tree_selection','main_ia_selection'])]
assert main['mode'].eq('fresh_fit').all()
fit_summary=main.groupby(['model','horizon_minutes','seed']).agg(component_fit_seconds=('fit_seconds','sum'),peak_process_rss_mb=('peak_rss_mb','max'),n_fit_calls=('fit_seconds','size')).reset_index()
inference=pd.read_csv(TABLE_DIR/'main_inference_calibration_runtime.csv')
test_runtime=inference.loc[inference['split'].fillna('test').eq('test')] if 'split' in inference else inference
inf=test_runtime.groupby(['model','horizon_minutes','seed']).agg(inference_seconds=('inference_seconds','sum'),n_inference=('n_inference','sum')).reset_index()
cal=inference.groupby(['model','horizon_minutes','seed']).calibration_seconds.sum().reset_index() if 'calibration_seconds' in inference else pd.DataFrame(columns=['model','horizon_minutes','seed','calibration_seconds'])
result=fit_summary.merge(inf,on=['model','horizon_minutes','seed'],how='outer').merge(cal,on=['model','horizon_minutes','seed'],how='left')
modern=pd.read_csv(TABLE_DIR/'modern_selected_validation_configs.csv')
modern=modern.rename(columns={'training_seconds':'selected_neural_training_seconds','n_parameters':'neural_trainable_parameters','peak_rss_mb':'neural_peak_process_rss_mb','device':'training_device'})
result=result.merge(modern[['model','horizon_minutes','seed','selected_neural_training_seconds','neural_trainable_parameters','epochs_run','best_validation_epoch','neural_peak_process_rss_mb','training_device','precision','peak_mps_driver_allocated_mb']],on=['model','horizon_minutes','seed'],how='outer')
neural_budget=pd.read_csv(TABLE_DIR/'modern_computational_budget.csv').groupby(['model','horizon_minutes','seed']).training_seconds.sum().rename('neural_selection_and_fit_seconds').reset_index()
result=result.merge(neural_budget,on=['model','horizon_minutes','seed'],how='outer')
conformal_et=result.loc[result.model.eq('ExtraTrees')].copy();conformal_et['model']='Conformalized ExtraTrees';conformal_et['fit_reuse_note']='Same ET point checkpoint; calibration cost is additional.'
result.loc[result.model.eq('ExtraTrees'),'calibration_seconds']=0.
result=pd.concat([result,conformal_et],ignore_index=True)
result.loc[result.model.isin(['Persistence','Seasonal Persistence']),'component_fit_seconds']=0.
result['benchmark_partition']='full_test'
foundation_path=TABLE_DIR/'q1_foundation_baseline_metrics.csv'
if foundation_path.exists():
    foundation=pd.read_csv(foundation_path);foundation['benchmark_partition']='foundation_matched_subset';foundation['component_fit_seconds']=0.;foundation['fit_reuse_note']='Zero-shot; separate pinned environment; no training performed.'
    foundation=foundation.rename(columns={'n_parameters':'neural_trainable_parameters','n_forecasts':'n_inference'})
    result=pd.concat([result,foundation],ignore_index=True)
state=json.loads((STATE_DIR/'main_preprocessing_state.json').read_text())
result['shared_feature_engineering_seconds']=state['feature_engineering_seconds'];result['shared_graph_construction_seconds']=state['graph_construction_seconds']
result['shared_cost_note']='One preprocessing/graph cost per experiment; do not sum this duplicated annotation across model rows.'
result['training_device']=result.training_device.fillna('cpu')
result['inference_device']='cpu'
result['fit_device_caveat']='CPU and MPS training times are different hardware scopes; no cross-device speed superiority claim.'
result.to_csv(TABLE_DIR/'runtime_complexity.csv',index=False)
cache=components.groupby(['scope','mode']).agg(calls=('fit_seconds','size'),fit_seconds=('fit_seconds','sum')).reset_index();cache.to_csv(TABLE_DIR/'cache_vs_fresh_runtime.csv',index=False)
display(result[['model','horizon_minutes','seed','component_fit_seconds','selected_neural_training_seconds','calibration_seconds','inference_seconds','peak_process_rss_mb']])
display(cache)


,model,horizon_minutes,seed,component_fit_seconds,selected_neural_training_seconds,calibration_seconds,inference_seconds,peak_process_rss_mb
0,DLinear-lite,15,11,NaN,20.988594,0.0,0.249976,NaN
1,DLinear-lite,15,23,NaN,14.378789,0.0,0.254395,NaN
2,DLinear-lite,15,37,NaN,17.621879,0.0,0.262550,NaN
3,DLinear-lite,15,53,NaN,22.461863,0.0,0.272869,NaN
4,DLinear-lite,15,71,NaN,28.627831,0.0,0.281947,NaN
...,...,...,...,...,...,...,...,...
125,Lag-Llama zero-shot,60,11,0.0,NaN,NaN,70.642676,NaN
126,Lag-Llama zero-shot,60,23,0.0,NaN,NaN,67.405552,NaN
127,Lag-Llama zero-shot,60,37,0.0,NaN,NaN,70.160576,NaN
128,Lag-Llama zero-shot,60,53,0.0,NaN,NaN,70.012358,NaN


,scope,mode,calls,fit_seconds
0,ablation,cached_deterministic_component,1670,0.000000
1,ablation,fresh_fit,890,1863.211037
2,main_ia_selection,fresh_fit,90,304.419991
3,main_tree_selection,fresh_fit,78,813.854745
4,public_supplement,cached_deterministic_component,96,0.000000
5,public_supplement,fresh_fit,164,911.461455
6,rolling,cached_deterministic_component,246,0.000000
7,rolling,fresh_fit,564,4619.731008


In [6]:
def estimator_size(model):
    if hasattr(model,'estimators_'):
        trees=[t for t in np.asarray(model.estimators_,dtype=object).flat if hasattr(t,'tree_')]
        return sum(t.tree_.node_count for t in trees),len(trees)
    if hasattr(model,'_predictors'):
        trees=[t for iteration in model._predictors for t in iteration]
        return sum(len(t.nodes) for t in trees),len(trees)
    return 0,0
sizes=[]
for path in sorted(MODEL_DIR.glob('*.joblib')):
    bundle=load_model(path)
    if not isinstance(bundle,dict):continue
    estimators={k:v for k,v in bundle.items() if hasattr(v,'predict')}
    if isinstance(bundle.get('quantiles'),dict):estimators.update({f'quantile_{q}':v for q,v in bundle['quantiles'].items()})
    seen=set()
    for branch,model in estimators.items():
        if id(model) in seen:continue
        seen.add(id(model))
        nodes,trees=estimator_size(model)
        sizes.append({'model_file':path.name,'horizon_minutes':15*bundle.get('h',0),'seed':bundle.get('seed'),'branch':branch,'estimator':type(model).__name__,'tree_nodes':nodes,'trees':trees,'serialized_bundle_bytes':path.stat().st_size,'interpretation':'Tree structure count, not neural parameters'})
    del bundle
pd.DataFrame(sizes).to_csv(TABLE_DIR/'tree_model_structure_counts.csv',index=False)
(STATE_DIR/'runtime_analysis_complete.json').write_text(json.dumps({'status':'COMPLETE','fresh_main_fit_and_inference':True,'cache_accounting_separate':True},indent=2))


In [2]:
def flatten_config(value,prefix=''):
    for key,item in value.items():
        path=f'{prefix}.{key}' if prefix else key
        if isinstance(item,dict):yield from flatten_config(item,path)
        else:yield path,json.dumps(item,default=str)
representative=joblib.load(MODEL_DIR/'ia_gtpf_h1_seed11.joblib')
shared_config=representative['config'];del representative
(STATE_DIR/'IA_GTPF_CONFIG.json').write_text(json.dumps(shared_config,indent=2,default=str))
parameters=pd.read_csv(TABLE_DIR/'ia_gtpf_final_hyperparameters.csv').to_dict('records')
for path in sorted(STATE_DIR.glob('ia_gtpf_h*_seed*_config.json')):
    cfg=json.loads(path.read_text())
    cfg['empirical_bounds_rule']=shared_config['empirical_bounds']
    for key,value in flatten_config({k:v for k,v in cfg.items() if k!='fitted_models'}):
        parameters.append({'horizon_minutes':cfg['horizon_minutes'],'seed':cfg['seed'],'branch':'pipeline','parameter':key,'value':value})
parameters=pd.DataFrame(parameters).drop_duplicates(['horizon_minutes','seed','branch','parameter'],keep='last')
parameters.to_csv(TABLE_DIR/'ia_gtpf_final_hyperparameters.csv',index=False)
